<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# xDeepFM : the eXtreme Deep Factorization Machine 
This notebook will give you a quick example of how to train an [xDeepFM model](https://arxiv.org/abs/1803.05170). 
xDeepFM \[1\] is a deep learning-based model aims at capturing both lower- and higher-order feature interactions for precise recommender systems. Thus it can learn feature interactions more effectively and manual feature engineering effort can be substantially reduced. To summarize, xDeepFM has the following key properties:
* It contains a component, named CIN, that learns feature interactions in an explicit fashion and in vector-wise level;
* It contains a traditional DNN component that learns feature interactions in an implicit fashion and in bit-wise level.
* The implementation makes this model quite configurable. We can enable different subsets of components by setting hyperparameters like `use_Linear_part`, `use_FM_part`, `use_CIN_part`, and `use_DNN_part`. For example, by enabling only the `use_Linear_part` and `use_FM_part`, we can get a classical FM model.

The default CPU run trains xDeepFM on labeled synthetic click records. Prepared Criteo FFM files can be supplied through local mode.

CPU first run uses explicitly synthetic field-aware click data, not Criteo measurements. The historical remote resource host is unavailable. For prepared local FFM data set XDEEPFM_DATA_SOURCE=local, XDEEPFM_DATA_DIR and XDEEPFM_FEATURE_COUNT.


## 0. Global Settings and Imports

In [ ]:
import os
import sys
import numpy as np
from tempfile import TemporaryDirectory
import tensorflow as tf
tf.get_logger().setLevel('ERROR') # only show error messages

from recommenders.models.deeprec.deeprec_utils import download_deeprec_resources, prepare_hparams
from recommenders.models.deeprec.models.xDeepFM import XDeepFMModel
from recommenders.models.deeprec.io.iterator import FFMTextIterator
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"Tensorflow version: {tf.__version__}")

#### Parameters

In [ ]:
EPOCHS = 10
BATCH_SIZE = 256
RANDOM_SEED = 42  # Set this to None for non-deterministic result


xDeepFM uses the FFM format as data input: `<label> <field_id>:<feature_id>:<feature_value>`  
Each line represents an instance, `<label>` is a binary value with 1 meaning positive instance and 0 meaning negative instance. 
Features are divided into fields. For example, user's gender is a field, it contains three possible values, i.e. male, female and unknown. Occupation can be another field, which contains many more possible values than the gender field. Both field index and feature index are starting from 1. <br>

In [ ]:
tmpdir = TemporaryDirectory()
data_path = tmpdir.name
train_file = os.path.join(data_path, "train.ffm")
valid_file = os.path.join(data_path, "valid.ffm")
test_file = os.path.join(data_path, "test.ffm")
DATA_SOURCE = os.environ.get("XDEEPFM_DATA_SOURCE", "synthetic")
FIELD_COUNT = 39
if DATA_SOURCE == "synthetic":
    FEATURE_COUNT = FIELD_COUNT * 8
    rng = np.random.default_rng(RANDOM_SEED)
    categories = rng.integers(0, 8, size=(2000, FIELD_COUNT))
    labels = ((categories[:, 0] % 2) ^ (rng.random(2000) < 0.1)).astype(int)
    for filename, indices in [(train_file, range(1600)), (valid_file, range(1600, 1800)),
                              (test_file, range(1800, 2000))]:
        with open(filename, "w") as stream:
            for row in indices:
                features = [f"{field+1}:{field*8+int(categories[row, field])+1}:1"
                            for field in range(FIELD_COUNT)]
                stream.write(str(labels[row]) + " " + " ".join(features) + "\n")
elif DATA_SOURCE == "local":
    data_path = os.environ["XDEEPFM_DATA_DIR"]
    train_file, valid_file, test_file = [os.path.join(data_path, name)
                                       for name in ("train.ffm", "valid.ffm", "test.ffm")]
    FEATURE_COUNT = int(os.environ["XDEEPFM_FEATURE_COUNT"])
else:
    raise ValueError("Choose synthetic or local")
print("Data source:", DATA_SOURCE)


## Prepare labeled click data

The default is 2,000 seeded synthetic records in FFM format with 39 fields.
The split contains 1,600 training, 200 validation and 200 test records.
These results describe synthetic execution only. Use local mode with legally
obtained, consistently encoded Criteo files for a real-data experiment.


In [ ]:
hparams = prepare_hparams(
    model_type="xDeepFM", FEATURE_COUNT=FEATURE_COUNT, FIELD_COUNT=FIELD_COUNT,
    dim=4, method="classification", loss="cross_entropy_loss", data_format="ffm",
    activation=["relu", "relu"], dropout=[0.0, 0.0], metrics=["auc", "logloss"],
    cross_l2=0.01, embed_l2=0.01, layer_l2=0.01, learning_rate=0.002,
    batch_size=BATCH_SIZE, epochs=EPOCHS, cross_layer_sizes=[20, 10],
    init_value=0.1, layer_sizes=[20, 20], use_Linear_part=True,
    use_CIN_part=True, use_DNN_part=True, save_model=False, show_step=1000,
)
print(hparams)


In [ ]:
model = XDeepFMModel(hparams, FFMTextIterator, seed=RANDOM_SEED)

In [ ]:
# check the predictive performance before the model is trained
print(model.run_eval(test_file)) 

In [ ]:
%%time
model.fit(train_file, valid_file)

In [ ]:
# check the predictive performance after the model is trained
result = model.run_eval(test_file)
print(result)
assert all(np.isfinite(v) for v in result.values())
assert 0 <= result["auc"] <= 1 and result["logloss"] >= 0


In [ ]:
# Record results for tests - ignore this cell
store_metadata("auc", result["auc"])
store_metadata("logloss", result["logloss"])
store_metadata("epochs", EPOCHS)


In [ ]:
# Close the session before deleting synthetic files.
model.sess.close()
tmpdir.cleanup()


## Reference
\[1\] Lian, J., Zhou, X., Zhang, F., Chen, Z., Xie, X., & Sun, G. (2018). xDeepFM: Combining Explicit and Implicit Feature Interactions for Recommender Systems. Proceedings of the 24th ACM SIGKDD International Conference on Knowledge Discovery \& Data Mining, KDD 2018, London, UK, August 19-23, 2018.<br>